# Домашнее задание: MLflow — практика управления ML-экспериментами

В этом домашнем задании вы на практике познакомитесь с MLflow — инструментом для управления жизненным циклом моделей машинного обучения.

**Важно**: Перед выполнением заданий убедитесь, что:
- Docker Compose запущен (`docker-compose up -d`)
- MLflow UI доступен по адресу: http://localhost:5050
- Файл `.env` заполнен вашими ключами доступа к S3

## Импорт библиотек

In [2]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, mean_squared_error
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Часть 1 (4 балла)

---

### Задание 1 (0.25 балла)

Настройте MLflow Tracking Server так, чтобы он смотрел на наш запущенный в той же docker-compose сети сервис `mlflow-service`

In [3]:
mlflow.set_tracking_uri("http://mlflow-service:5000")


print(f"MLflow Tracking URI установлен: {mlflow.get_tracking_uri()}")
print(f"Проверьте доступность MLflow UI: http://localhost:5050")

✓ MLflow Tracking URI установлен: http://mlflow-service:5000
✓ Проверьте доступность MLflow UI: http://localhost:5050


### Задание 2 (0.25 балла)

Создайте эксперимент с именем `Surname_Name` (подставьте ваши Имя и Фамилию).

Обеспечьте проверку на уже созданный эксперимент, чтобы код не падал с ошибками, если эксперимент уже создан.

In [4]:
experiment_name = "trichev_andrey" 

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Создан новый эксперимент: '{experiment_name}'")
    print(f"ID эксперимента: {experiment_id}")
else:
    experiment_id = experiment.experiment_id
    print(f"Используется существующий эксперимент: '{experiment_name}'")
    print(f"ID эксперимента: {experiment_id}")

mlflow.set_experiment(experiment_name)

Создан новый эксперимент: 'trichev_andrey'
ID эксперимента: 545032767681248927


<Experiment: artifact_location='s3://mlflow/artifacts/545032767681248927', creation_time=1766608133265, experiment_id='545032767681248927', last_update_time=1766608133265, lifecycle_stage='active', name='trichev_andrey', tags={}>

### Загрузка и подготовка датасета

Используем датасет Wine Classification из sklearn

In [5]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Задание 3 (1 балл)

Используйте датасет Wine (уже загружен выше).

Обучите простую модель LogisticRegression.

Заведите `mlflow.start_run()` и залогируйте вручную:
- параметры модели (0.25б)
- метрики (Accuracy) (0.25б)
- артефакт: CSV-файл с предсказаниями (0.25б)
- `run_name="your_telegram_nickname"` - укажите свой ник в Telegram (0.25б

In [6]:

with mlflow.start_run(run_name="trichev03") as run:
    model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    mlflow.log_param("C", 1.0)
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("random_state", 42)
    
    mlflow.log_metric("accuracy", accuracy)
    
    predictions_df = pd.DataFrame({
        'actual': y_test.values,
        'predicted': predictions
    })
    predictions_df.to_csv('predictions.csv', index=False)
    mlflow.log_artifact('predictions.csv')


2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run trichev03 at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/910652e56bf3454f8a702f1652e6c931.
2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.


### Задание 4 (1 балл)

Сравните несколько моделей через MLflow (GridSearch вручную).

- Переберите минимум 4 набора гиперпараметров (0.25б)
- Для каждого варианта создайте новый run (0.25б)
- Добавьте теги `model_type=ridge` и `stage=research` (0.25б)
- В конце выведите таблицу: параметры — метрика — run_id (0.25б)

In [7]:

alphas = [0.1, 1.0, 10.0, 100.0]
results = []

for alpha in alphas:

    with mlflow.start_run(run_name=f"ridge_alpha_{alpha}") as run:

        mlflow.set_tag("model_type", "ridge")
        mlflow.set_tag("stage", "research")
        

        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_train, y_train)
        

        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)

        mlflow.log_param("alpha", alpha)
        mlflow.log_param("model_type", "Ridge")
        mlflow.log_metric("mse", mse)
        
        results.append({
            'alpha': alpha,
            'mse': mse,
            'run_id': run.info.run_id
        })
        
        print(f"  Alpha: {alpha:6.1f} | MSE: {mse:.4f} | Run ID: {run.info.run_id}")


results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run ridge_alpha_0.1 at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/68bf112df09043b7af624d5b75f99f07.
2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.
2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run ridge_alpha_1.0 at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/dfce767571644935a5ff0a574712a65b.
2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.
2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run ridge_alpha_10.0 at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/559b2bd1bd004b4d861c67c389f186f7.
2025/12/24 20:28:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-servi

  Alpha:    0.1 | MSE: 0.0685 | Run ID: 68bf112df09043b7af624d5b75f99f07
  Alpha:    1.0 | MSE: 0.0681 | Run ID: dfce767571644935a5ff0a574712a65b
  Alpha:   10.0 | MSE: 0.0693 | Run ID: 559b2bd1bd004b4d861c67c389f186f7
  Alpha:  100.0 | MSE: 0.0909 | Run ID: 04e06a324fc4450a959a700caebc58f2
 alpha      mse                           run_id
   0.1 0.068458 68bf112df09043b7af624d5b75f99f07
   1.0 0.068102 dfce767571644935a5ff0a574712a65b
  10.0 0.069342 559b2bd1bd004b4d861c67c389f186f7
 100.0 0.090937 04e06a324fc4450a959a700caebc58f2


### Задание 5 (0.25 балла)

Используйте autologging.

- Включите `mlflow.sklearn.autolog()`
- Обучите sklearn-пайплайн (StandardScaler + RandomForest)
- Убедитесь, что:
  - параметры RF залогированы автоматически
  - важность признаков сохранена как артефакт
  - модель записана в MLflow

In [8]:

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="autolog_random_forest") as run:
   
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
    ])
    

    pipeline.fit(X_train, y_train)
    
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    
mlflow.sklearn.autolog(disable=True)

2025/12/24 20:28:56 INFO mlflow.tracking._tracking_service.client: 🏃 View run autolog_random_forest at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/28ef170811db4fe99f995b4e8a0e8ed7.
2025/12/24 20:28:56 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.


### Задание 6 (0.5 балла)

Nested runs: логирование подэкспериментов.

Постройте структуру:
```
main_run
 ├── model_lr
 ├── model_rf
 └── model_gb
```

- Внутри одного `main_run` создайте три дочерних run'а (0.25б)
- В `main_run` залогируйте метрику: лучшая accuracy среди трёх (0.25б)

In [9]:

with mlflow.start_run(run_name="main_comparison") as main_run:
    best_accuracy = 0
    best_model_name = ""
    
    models_config = [
        ('model_lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('model_rf', RandomForestClassifier(n_estimators=50, random_state=42)),
        ('model_gb', GradientBoostingClassifier(n_estimators=50, random_state=42))
    ]
    
    for model_name, model in models_config:
        with mlflow.start_run(run_name=model_name, nested=True) as child_run:
        
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            
            
            mlflow.log_param("model_type", model_name)
            mlflow.log_metric("accuracy", accuracy)
            
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_model_name = model_name

    mlflow.log_metric("best_accuracy", best_accuracy)
    mlflow.log_param("best_model", best_model_name)
    
    print(f"  Лучшая модель: {best_model_name}")
    print(f"  Лучшая accuracy: {best_accuracy:.4f}")


2025/12/24 20:28:56 INFO mlflow.tracking._tracking_service.client: 🏃 View run model_lr at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/5adcc1f30a8742418e53d19d75afd1ac.
2025/12/24 20:28:56 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.
2025/12/24 20:28:57 INFO mlflow.tracking._tracking_service.client: 🏃 View run model_rf at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/cac2e326ed2d4ee88d43e1ba04303684.
2025/12/24 20:28:57 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.
2025/12/24 20:28:57 INFO mlflow.tracking._tracking_service.client: 🏃 View run model_gb at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/26160e2af34845ac965ee3163f897ea7.
2025/12/24 20:28:57 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/

  Лучшая модель: model_rf
  Лучшая accuracy: 1.0000


### Задание 7 (0.5 балла)

MLflow Model Registry — регистрация и промоут модели.

- Выберите лучшую модель из предыдущих run'ов
- Зарегистрируйте её в Model Registry под именем `best_wine_model`
- Промоутите модель в стадию **Staging** (0.25б)
- Обновите описание модели (0.25б):
  - используемый датасет
  - дата
  - параметры

In [10]:

with mlflow.start_run(run_name="best_model_for_registry") as run:

    best_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    best_model.fit(X_train, y_train)
    
    predictions = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("model_class", "RandomForestClassifier")
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.sklearn.log_model(best_model, "model")
    
    best_run_id = run.info.run_id
    print(f"Модель обучена:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Run ID: {best_run_id}")

model_name = "best_wine_model"
model_uri = f"runs:/{best_run_id}/model"

try:
    
    registered_model = mlflow.register_model(model_uri, model_name)
    

    client = mlflow.tracking.MlflowClient()
    client.transition_model_version_stage(
        name=model_name,
        version=registered_model.version,
        stage="Staging"
    )
    
   
    description = f"""Датасет: Wine Classification
Дата: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Параметры: 
  - n_estimators: 100
  - max_depth: 10
  - random_state: 42
Метрики:
  - Accuracy: {accuracy:.4f}
Автор: {experiment_name}
"""
    
    client.update_model_version(
        name=model_name,
        version=registered_model.version,
        description=description
    )
    
    
    
except Exception as e:
    print(e)
    

2025/12/24 20:28:58 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/12/24 20:28:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run best_model_for_registry at: http://mlflow-service:5000/#/experiments/545032767681248927/runs/1cdf9c00af654aa6ba6848e02a3993a8.
2025/12/24 20:28:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/545032767681248927.
Successfully registered model 'best_wine_model'.
2025/12/24 20:28:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_wine_model, version 1


Модель обучена:
  Accuracy: 1.0000
  Run ID: 1cdf9c00af654aa6ba6848e02a3993a8


Created version '1' of model 'best_wine_model'.


### Задание 8 (0.25 балла)

Загрузка и инференс модели из Registry.

- Через Python загрузите версию модели из Registry (0.125б)
- Сделайте предсказания на тестовом наборе (0.125б)

In [11]:

try:
    
    loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}/Staging")
    
   
    test_predictions = loaded_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_predictions)
    
    
    print(f"  Источник: models:/{model_name}/Staging")
    print(f"  Accuracy на тестовых данных: {test_accuracy:.4f}")
    
    
    comparison_df = pd.DataFrame({
        'Реальный класс': y_test[:10].values,
        'Предсказание': test_predictions[:10]
    })
    print(comparison_df.to_string(index=False))
    
    
    print(f"  Всего предсказаний: {len(test_predictions)}")
    print(f"  Правильных: {(test_predictions == y_test).sum()}")
    print(f"  Ошибок: {(test_predictions != y_test).sum()}")
    
except Exception as e:
    print(e)
    

  Источник: models:/best_wine_model/Staging
  Accuracy на тестовых данных: 1.0000
 Реальный класс  Предсказание
              0             0
              0             0
              2             2
              0             0
              1             1
              0             0
              1             1
              2             2
              1             1
              2             2
  Всего предсказаний: 36
  Правильных: 36
  Ошибок: 0
